In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")
print(f"Environment: {env}")

catalog = f"saleslake_{env}"
bronzeSchema = f"bronze_schema_{env}"

targetTable    = f"{catalog}.{bronzeSchema}.rawInvoice"
srcFileLoc  = f"/Volumes/{catalog}/{bronzeSchema}/vol_saleslake_src_files_{env}/invoice/"
print(f"Target table : {targetTable}")
print(f"Source files : {srcFileLoc}")

In [0]:
spark.sql(f"""
COPY INTO {targetTable}
FROM (
    SELECT invoice_id,invoice_number,customer_id,invoice_date,due_date,subtotal_amount,discount_code,discount_amount,tax_amount,total_amount,payment_status,payment_method,payment_date,currency,region,store_id,channel,created_by,current_timestamp AS ingest_ts
    FROM '{srcFileLoc}'
    )
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")
print("Invoice Bronze Load Successful")

In [0]:
spark.sql(f"""
        SELECT COUNT(*) AS total_records
        FROM {targetTable}
    """).show()